# DockBench — Inference (Kaggle / Colab)

**Add Data:** `duchvminh/results` (weights + benchmark CSV)  
**Demo:** `code_docking/demo_inference/` trong repo (đã có `types/demo.types` + `.gninatypes`) — hoặc upload lên Kaggle Dataset

Settings: **GPU**, **Internet**, **Run All**.

## 1. Cấu hình

In [ ]:
from pathlib import Path

# Kaggle hoặc Colab
WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")

REPO_URL = "https://github.com/nWoWolfpac/KhoaLuanTotNghiep.git"
BRANCH = "main"
CLONE_DIR = WORKDIR / "KhoaLuanTotNghiep"

MODEL_ID = "geoformerdock"
CHECKPOINT = None

# --- Results (models + plots) ---
KAGGLE_RESULTS_PATH = "/kaggle/input/datasets/duchvminh/results/results"
DRIVE_RESULTS_FOLDER_ID = ""
RESULTS_CACHE = WORKDIR / "results_from_drive"

# --- Demo (đã có trong repo: demo_inference/) ---
# None = sau §2 dùng DOCK_ROOT/demo_inference; hoặc path Kaggle Dataset
DEMO_DATA_PATH = None
KAGGLE_DEMO_DATA_ROOT = "/kaggle/input/datasets/duchvminh/demo-inference"
DRIVE_DEMO_FOLDER_ID = ""
DEMO_CACHE = WORKDIR / "demo_from_drive"
DEMO_TYPES_FILE = "types/demo.types"
DEMO_COMPLEX_IDS = None  # None = mọi dòng trong demo.types (3zsx + 4eky)

METRICS_CSV_REL = "log_dynamics/baseline_comparison/compare_best_metrics.csv"

## 2. Clone repo + cài phụ thuộc

In [ ]:
import os
import sys
import json
import subprocess

def sh(cmd, check=True):
    print(">>>", cmd)
    return subprocess.run(cmd, shell=True, check=check)

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    subprocess.run(cmd, check=True)

clone = Path(CLONE_DIR)
if not (clone / ".git").is_dir():
    sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {clone}")
else:
    sh(f"cd {clone} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only", check=False)

if (clone / "code_docking" / "dockbench").is_dir():
    DOCK_ROOT = clone / "code_docking"
elif (clone / "dockbench").is_dir():
    DOCK_ROOT = clone
else:
    raise FileNotFoundError(f"Không thấy dockbench trong {clone}")

sys.path.insert(0, str(DOCK_ROOT))
os.chdir(DOCK_ROOT)
print("DOCK_ROOT =", DOCK_ROOT.resolve())

pip_install("numpy>=1.26,<2")
pip_install("torch==2.2.2", "pytorch-ignite>=0.4.10", "PyYAML>=5.4", "tqdm>=4.62")
pip_install("molgrid")
print("OK: numpy + torch + molgrid (restart runtime nếu Colab báo lỗi NumPy 2)")

## 3. Load `results/` (weights + benchmark CSV)

In [ ]:
def _discover_results_root():
    inp = Path("/kaggle/input")
    if not inp.is_dir():
        return None
    best = None
    for models_dir in inp.rglob("models"):
        if models_dir.is_dir() and list(models_dir.rglob("best_model.pt")):
            root = models_dir.parent
            if best is None or "duchvminh" in str(root):
                best = root
    return best

def load_results_root():
    if KAGGLE_RESULTS_PATH:
        p = Path(KAGGLE_RESULTS_PATH)
        if p.is_dir():
            return p.resolve()
    found = _discover_results_root()
    if found is not None:
        return found.resolve()
    if DRIVE_RESULTS_FOLDER_ID:
        pip_install("gdown")
        cache = Path(RESULTS_CACHE)
        if not (cache / ".ok").is_file():
            cache.mkdir(parents=True, exist_ok=True)
            sh(f"gdown --folder https://drive.google.com/drive/folders/{DRIVE_RESULTS_FOLDER_ID} -O {cache} --remaining-ok", check=False)
            (cache / ".ok").write_text("ok")
        for sub in [cache, cache / "results"]:
            if sub.is_dir() and list(sub.rglob("best_model.pt")):
                return sub.resolve()
    raise FileNotFoundError("Add Data duchvminh/results hoặc đặt KAGGLE_RESULTS_PATH")

RESULTS_ROOT = load_results_root()
models_root = RESULTS_ROOT / "models" if (RESULTS_ROOT / "models").is_dir() else RESULTS_ROOT
plots_dir = RESULTS_ROOT / "plots" if (RESULTS_ROOT / "plots" / METRICS_CSV_REL).is_file() else RESULTS_ROOT

if not list(models_root.rglob("best_model.pt")):
    raise FileNotFoundError(f"Không có best_model.pt trong {models_root}")
if not (plots_dir / METRICS_CSV_REL).is_file():
    raise FileNotFoundError(f"Thiếu {METRICS_CSV_REL}")

print("RESULTS_ROOT =", RESULTS_ROOT)
print("models_root  =", models_root)
print("plots_dir    =", plots_dir)
print("checkpoints  =", len(list(models_root.rglob("best_model.pt"))))

## 4. Benchmark huấn luyện (CSV)

In [ ]:
import csv

csv_path = plots_dir / METRICS_CSV_REL
with csv_path.open(newline="", encoding="utf-8") as f:
    bench_rows = list(csv.DictReader(f))

cols = [c for c in [
    "display_model", "model", "best_c_index", "best_balanced_acc",
    "best_pearson_r", "best_pr_auc", "best_mae", "best_rmse",
] if bench_rows and c in bench_rows[0]]

if "best_c_index" in cols:
    bench_rows.sort(key=lambda r: float(r.get("best_c_index") or -1e9), reverse=True)

print(f"=== Benchmark train ({csv_path.name}) ===")
print(" | ".join(cols))
for r in bench_rows:
    print(" | ".join(str(r.get(c, "")) for c in cols))

## 5. Load model + demo data

In [ ]:
import shutil
from typing import Tuple

import torch
import molgrid
from dockbench.models.registry import build_model, canonical_name
from dockbench.target_normalizer import TargetNormalizer

print("torch", torch.__version__, "CUDA:", torch.cuda.is_available())
print("molgrid OK")


def _summary_near_ckpt(ckpt: Path) -> dict:
    for p in [ckpt.parent / "summary.json", ckpt.parent.parent / "summary.json"]:
        if p.is_file():
            return json.loads(p.read_text(encoding="utf-8"))
    return {}


def find_checkpoint(model_id: str) -> Path:
    if CHECKPOINT:
        return Path(CHECKPOINT).resolve()
    hits = [p for p in models_root.rglob("best_model.pt") if model_id in p.as_posix()]
    if not hits:
        hits = list((models_root / model_id).rglob("best_model.pt"))
    if not hits:
        raise FileNotFoundError(f"Không có best_model.pt cho {model_id}")
    return sorted(hits)[-1].resolve()


def load_model(ckpt_path: Path, input_dims: Tuple[int, int, int, int]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    name = canonical_name(payload.get("model", MODEL_ID))
    summary = _summary_near_ckpt(ckpt_path)
    gkw = None
    if name == "geoformerdock":
        gkw = {
            "max_pseudo_atoms": int(summary.get("max_pseudo_atoms", 12)),
            "num_transformer_layers": int(summary.get("num_transformer_layers", 2)),
            "uncertainty": bool(summary.get("geoformer_uncertainty", False)),
        }
    model = build_model(name, input_dims, affinity=True, flex=False, geoformer_kwargs=gkw)
    model.load_state_dict(payload["model_state_dict"])
    model.to(device).eval()
    norm = None
    ns = payload.get("target_normalizer")
    if payload.get("normalize_targets") and ns:
        norm = TargetNormalizer()
        norm.mean, norm.std, norm.fitted = float(ns["mean"]), float(ns["std"]), True
    return model, device, norm, name


def load_demo_root():
    candidates = []
    if DEMO_DATA_PATH:
        candidates.append(Path(DEMO_DATA_PATH))
    repo_demo = DOCK_ROOT / "demo_inference"
    candidates.append(repo_demo)
    if KAGGLE_DEMO_DATA_ROOT:
        candidates.append(Path(KAGGLE_DEMO_DATA_ROOT))
    inp = Path("/kaggle/input")
    if inp.is_dir():
        for name in ("demo-inference", "demo_inference", "demo"):
            for p in inp.rglob(name):
                if (p / "types" / "demo.types").is_file() or list(p.rglob("*.gninatypes")):
                    candidates.append(p)
    for p in candidates:
        p = p.resolve()
        types = p / DEMO_TYPES_FILE
        if not types.is_file():
            hits = list(p.rglob("demo.types"))
            if hits:
                types = hits[0]
        if types.is_file() and list(p.rglob("*.gninatypes")):
            print("demo_root =", p)
            return p
    if DRIVE_DEMO_FOLDER_ID:
        pip_install("gdown")
        cache = Path(DEMO_CACHE)
        if not (cache / ".ok").is_file():
            cache.mkdir(parents=True, exist_ok=True)
            sh(f"gdown --folder https://drive.google.com/drive/folders/{DRIVE_DEMO_FOLDER_ID} -O {cache} --remaining-ok", check=False)
            (cache / ".ok").write_text("ok")
        for sub in [cache, cache / "demo_inference"]:
            if sub.is_dir() and list(sub.rglob("*.gninatypes")):
                return sub.resolve()
    raise FileNotFoundError(
        f"Không thấy demo_inference (cần types/demo.types + .gninatypes). "
        f"Đã thử: {repo_demo}"
    )


demo_root = load_demo_root()
types_path = demo_root / DEMO_TYPES_FILE
if not types_path.is_file():
    hits = list(demo_root.rglob("demo.types"))
    if not hits:
        raise FileNotFoundError("Không thấy demo.types")
    types_path = hits[0]
    if types_path.parent.name == "types":
        demo_root = types_path.parent.parent

filter_ids = {x.lower() for x in DEMO_COMPLEX_IDS} if DEMO_COMPLEX_IDS else None
DEMO_RUNS = []
for raw in types_path.read_text(encoding="utf-8").splitlines():
    line = raw.split("#", 1)[0].strip()
    if not line:
        continue
    parts = line.split()
    if len(parts) < 4:
        continue
    rec_rel, lig_rel = Path(parts[-2]), Path(parts[-1])
    cid = rec_rel.parts[0].lower()
    if filter_ids and cid not in filter_ids:
        continue
    rec_src, lig_src = demo_root / rec_rel, demo_root / lig_rel
    if not rec_src.is_file() or not lig_src.is_file():
        raise FileNotFoundError(f"Thiếu {rec_src} hoặc {lig_src}")
    work = WORKDIR / "dockbench_work" / cid
    work.mkdir(parents=True, exist_ok=True)
    shutil.copy2(rec_src, work / "rec.gninatypes")
    shutil.copy2(lig_src, work / "lig.gninatypes")
    (work / "score.types").write_text(
        f"{parts[0]} {parts[1]} rec.gninatypes lig.gninatypes\n", encoding="utf-8"
    )
    DEMO_RUNS.append({"id": cid, "work": work, "pose_label": parts[0], "aff_label": parts[1]})

if not DEMO_RUNS:
    raise ValueError("Không có complex trong demo.types")

ckpt_path = find_checkpoint(MODEL_ID)
print("Checkpoint:", ckpt_path)
print("Demo complexes:", [r["id"] for r in DEMO_RUNS])

## 6. Inference (molgrid → model)

In [ ]:
gmaker = molgrid.GridMaker(resolution=0.5, dimension=23.5)

_probe = DEMO_RUNS[0]["work"]
_prov = molgrid.ExampleProvider(
    data_root=str(_probe), balanced=False, shuffle=False,
    default_batch_size=1, iteration_scheme=molgrid.IterationScheme.SmallEpoch,
    cache_structs=False,
)
_prov.populate(str(_probe / "score.types"))
INPUT_DIMS = tuple(int(x) for x in gmaker.grid_dimensions(_prov.num_types()))

model, device, normalizer, model_name = load_model(ckpt_path, INPUT_DIMS)
print(f"Model: {model_name}  dims: {INPUT_DIMS}\n")

print("=== Inference demo ===")
print(f"{'complex':<8} {'pose_label':<6} {'aff_types':<10} {'P(good)':<10} {'pK_pred':<10}")
print("-" * 50)

for run in DEMO_RUNS:
    work = run["work"]
    prov = molgrid.ExampleProvider(
        data_root=str(work), balanced=False, shuffle=False,
        default_batch_size=1, iteration_scheme=molgrid.IterationScheme.SmallEpoch,
        cache_structs=False,
    )
    prov.populate(str(work / "score.types"))
    grid = torch.zeros((1,) + INPUT_DIMS, dtype=torch.float32, device=device)
    gmaker.forward(prov.next_batch(1), grid, random_translation=0.0, random_rotation=False)
    with torch.no_grad():
        pose_log, aff = model(grid)
    pose_prob = float(torch.exp(pose_log)[0, 1].item())
    aff_val = float(aff[0].item())
    if normalizer and normalizer.fitted:
        aff_val = float(normalizer.denormalize(aff)[0].item())
    print(
        f"{run['id']:<8} {run['pose_label']:<6} {run['aff_label']:<10} "
        f"{pose_prob:<10.4f} {aff_val:<10.4f}"
    )

print("\nDone.")